In [ ]:
!pip install scikit-fuzzy geopy pandas openpyxl

import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from geopy.distance import geodesic

# =========================
# LOAD DATA EXCEL
# =========================
df = pd.read_excel("AI_dataset_HW_week2.xlsx")
df[["lat", "lon"]] = df["Tọa độ"].str.split(",", expand=True)

df["lat"] = df["lat"].astype(float)
df["lon"] = df["lon"].astype(float)
df = df[["Tên Nhà Hàng", "lat", "lon"]]
# =========================
# USER INPUT
# =========================
user_location = (10.7625, 106.6820)
user_weather = 25

# =========================
# FUZZY VARIABLES
# =========================
distance = ctrl.Antecedent(np.arange(0, 21, 1), "distance")
weather = ctrl.Antecedent(np.arange(15, 46, 1), "weather")

delivery_fee = ctrl.Consequent(np.arange(0, 100001, 1000), "delivery_fee")
delivery_time = ctrl.Consequent(np.arange(0, 91, 1), "delivery_time")

# =========================
# MEMBERSHIP FUNCTIONS
# =========================
distance["near"] = fuzz.trapmf(distance.universe, [0, 0, 1, 3])
distance["medium"] = fuzz.trimf(distance.universe, [2, 5, 8])
distance["far"] = fuzz.trapmf(distance.universe, [6, 10, 20, 20])

weather["cold"] = fuzz.trapmf(weather.universe, [15, 15, 18, 23])
weather["normal"] = fuzz.trimf(weather.universe, [22, 27, 32])
weather["hot"] = fuzz.trapmf(weather.universe, [30, 35, 45, 45])

delivery_fee["cheap"] = fuzz.trapmf(delivery_fee.universe, [0, 0, 10000, 20000])
delivery_fee["medium"] = fuzz.trimf(delivery_fee.universe, [15000, 30000, 50000])
delivery_fee["expensive"] = fuzz.trapmf(delivery_fee.universe, [45000, 60000, 100000, 100000])

delivery_time["fast"] = fuzz.trapmf(delivery_time.universe, [0, 0, 10, 17])
delivery_time["normal"] = fuzz.trimf(delivery_time.universe, [15, 25, 35])
delivery_time["slow"] = fuzz.trapmf(delivery_time.universe, [30, 40, 60, 60])

# =========================
# FUZZY RULES
# =========================
rule1 = ctrl.Rule(distance["near"], delivery_fee["cheap"])
rule2 = ctrl.Rule(distance["medium"], delivery_fee["medium"])
rule3 = ctrl.Rule(distance["far"], delivery_fee["expensive"])

rule4 = ctrl.Rule(distance["near"], delivery_time["fast"])
rule5 = ctrl.Rule(distance["medium"], delivery_time["normal"])
rule6 = ctrl.Rule(distance["far"], delivery_time["slow"])

rule7 = ctrl.Rule(weather["hot"] & distance["far"], delivery_fee["expensive"])
rule8 = ctrl.Rule(weather["hot"] & distance["far"], delivery_time["slow"])

rule9 = ctrl.Rule(weather["cold"] & distance["near"], delivery_fee["cheap"])
rule10 = ctrl.Rule(weather["cold"] & distance["near"], delivery_time["fast"])

rule11 = ctrl.Rule(weather["normal"] & distance["medium"], delivery_fee["medium"])
rule12 = ctrl.Rule(weather["normal"] & distance["medium"], delivery_time["normal"])

rule13 = ctrl.Rule(weather["hot"] & distance["medium"], delivery_fee["medium"])
rule14 = ctrl.Rule(weather["hot"] & distance["medium"], delivery_time["normal"])

rule15 = ctrl.Rule(weather["cold"] & distance["far"], delivery_fee["medium"])
rule16 = ctrl.Rule(weather["cold"] & distance["far"], delivery_time["slow"])

# =========================
# CONTROL SYSTEM
# =========================
fee_ctrl = ctrl.ControlSystem([
    rule1, rule2, rule3,
    rule7, rule9, rule11,
    rule13, rule15
])

time_ctrl = ctrl.ControlSystem([
    rule4, rule5, rule6,
    rule8, rule10, rule12,
    rule14, rule16
])

# =========================
# FUNCTION CALCULATE 1 RESTAURANT
# =========================
def evaluate(row):

    dist = geodesic(user_location, (row["lat"], row["lon"])).km

    # fee
    fee_sim = ctrl.ControlSystemSimulation(fee_ctrl)
    fee_sim.input["distance"] = dist
    fee_sim.input["weather"] = user_weather
    fee_sim.compute()

    # time
    time_sim = ctrl.ControlSystemSimulation(time_ctrl)
    time_sim.input["distance"] = dist
    time_sim.input["weather"] = user_weather
    time_sim.compute()

    return {
        "name": row["Tên Nhà Hàng"],
        "distance": dist,
        "fee": fee_sim.output["delivery_fee"],
        "time": time_sim.output["delivery_time"]
    }

# =========================
# RUN FOR ALL RESTAURANTS
# =========================
results = df.apply(evaluate, axis=1).tolist()
results_df = pd.DataFrame(results)

# =========================
# SCORE RANKING
# =========================
results_df["score"] = (
    100
  - results_df["fee"] / 1000
  - results_df["distance"] * 10
  - results_df["time"])

results_df = results_df.sort_values("score", ascending=False)

print(results_df)

ValueError: Point coordinates must be finite. (nan, nan, 0.0) has been passed as coordinates.